# 🥈 ETL Bronze → Silver

**Objectif** : Nettoyer et transformer les données brutes pour créer la couche Silver

## Transformations appliquées

1. **Filtrage géographique** : Petite Couronne uniquement (75, 92, 93, 94)
2. **Standardisation codes INSEE** : Toujours 5 caractères (padding zéros)
3. **Harmonisation colonnes** : Noms cohérents entre datasets
4. **Gestion valeurs manquantes** : Stratégies par dataset
5. **Création clés de jointure** : `code_insee` + `annee`
6. **Validation qualité** : Vérifier cohérence géographique et temporelle

---

## 1️⃣ Configuration et Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✅ Imports chargés")

✅ Imports chargés


In [2]:
# Chemins Medallion Architecture
notebook_dir = Path.cwd()
PROJECT_ROOT = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

DATA_BRONZE = PROJECT_ROOT / 'data' / 'bronze'
DATA_SILVER = PROJECT_ROOT / 'data' / 'silver'
DATA_GOLD = PROJECT_ROOT / 'data' / 'gold'

# Créer dossier Silver si inexistant
DATA_SILVER.mkdir(parents=True, exist_ok=True)

# Paramètres
DEPARTEMENTS_PC = ['75', '92', '93', '94']

print(f"🥉 Bronze : {DATA_BRONZE}")
print(f"🥈 Silver : {DATA_SILVER}")
print(f"🎯 Périmètre : {', '.join(DEPARTEMENTS_PC)}")

🥉 Bronze : /app/data/bronze
🥈 Silver : /app/data/silver
🎯 Périmètre : 75, 92, 93, 94


## 2️⃣ Référentiel Géographique : COG 2024

In [3]:
# Charger COG 2024 (référentiel maître)
cog_2024_path = DATA_BRONZE / "referentiels_cog" / "referentiel_communes_2024.csv"

print("📥 Chargement COG 2024...")
df_cog = pd.read_csv(cog_2024_path)

print(f"✅ {len(df_cog):,} communes chargées")
print(f"\nColonnes : {list(df_cog.columns)}")
print(f"\nÉchantillon :")
df_cog.head()

📥 Chargement COG 2024...
✅ 37,544 communes chargées

Colonnes : ['TYPECOM', 'COM', 'REG', 'DEP', 'CTCD', 'ARR', 'TNCC', 'NCC', 'NCCENR', 'LIBELLE', 'CAN', 'COMPARENT']

Échantillon :


,TYPECOM,COM,REG,DEP,CTCD,ARR,TNCC,NCC,NCCENR,LIBELLE,CAN,COMPARENT
0,COM,01001,84.0,01,01D,012,5,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,L'Abergement-Clémenciat,0108,NaN
1,COM,01002,84.0,01,01D,011,5,ABERGEMENT DE VAREY,Abergement-de-Varey,L'Abergement-de-Varey,0101,NaN
2,COM,01004,84.0,01,01D,011,1,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,Ambérieu-en-Bugey,0101,NaN
3,COM,01005,84.0,01,01D,012,1,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,Ambérieux-en-Dombes,0122,NaN
4,COM,01006,84.0,01,01D,011,1,AMBLEON,Ambléon,Ambléon,0104,NaN


In [4]:
# Identifier colonne code commune
# COG Insee utilise généralement 'COM' ou 'CODGEO'
code_col = None
for col in ['COM', 'CODGEO', 'code_commune', 'INSEE_COM']:
    if col in df_cog.columns:
        code_col = col
        break

if code_col is None:
    print("❌ Colonne code commune non trouvée")
    print(f"Colonnes disponibles : {list(df_cog.columns)}")
else:
    print(f"✅ Colonne code commune : {code_col}")
    
    # Standardiser code INSEE
    df_cog['code_insee'] = df_cog[code_col].astype(str).str.zfill(5)
    
    # Extraire département
    df_cog['dept'] = df_cog['code_insee'].str[:2]
    
    print(f"\nÉchantillon codes standardisés :")
    print(df_cog[['code_insee', 'dept']].head())

✅ Colonne code commune : COM

Échantillon codes standardisés :
  code_insee dept
0      01001   01
1      01002   01
2      01004   01
3      01005   01
4      01006   01


In [5]:
# Filtrer Petite Couronne
df_cog_pc = df_cog[df_cog['dept'].isin(DEPARTEMENTS_PC)].copy()

print("🎯 COMMUNES PETITE COURONNE")
print("=" * 60)
print(f"Total communes : {len(df_cog_pc)}")
print(f"\nRépartition par département :")
dept_counts = df_cog_pc.groupby('dept').size().sort_index()
for dept, count in dept_counts.items():
    print(f"  {dept} : {count} communes")

# Créer liste codes INSEE PC pour filtrage
codes_insee_pc = set(df_cog_pc['code_insee'].tolist())
print(f"\n💾 {len(codes_insee_pc)} codes INSEE Petite Couronne identifiés")

🎯 COMMUNES PETITE COURONNE
Total communes : 144

Répartition par département :
  75 : 21 communes
  92 : 36 communes
  93 : 40 communes
  94 : 47 communes

💾 144 codes INSEE Petite Couronne identifiés


In [6]:
# Sauvegarder référentiel PC en Silver
output_cog = DATA_SILVER / "referentiel_communes_pc_2024.parquet"

# Sélectionner colonnes utiles
cols_keep = ['code_insee', 'dept']
if 'LIBELLE' in df_cog_pc.columns:
    cols_keep.append('LIBELLE')
elif 'NCC' in df_cog_pc.columns:
    cols_keep.append('NCC')

df_cog_pc_clean = df_cog_pc[cols_keep].copy()
df_cog_pc_clean.to_parquet(output_cog, index=False)

print(f"✅ Référentiel sauvegardé : {output_cog.name}")
print(f"   Taille : {output_cog.stat().st_size / 1024:.2f} KB")

✅ Référentiel sauvegardé : referentiel_communes_pc_2024.parquet
   Taille : 4.77 KB


## 3️⃣ Naissances 2008-2024

In [7]:
# Trouver fichier naissances
naissances_dir = DATA_BRONZE / "naissances_2008_2024"
csv_files = list(naissances_dir.glob("*.csv"))

if not csv_files:
    print("❌ Aucun fichier CSV naissances trouvé")
else:
    csv_file = csv_files[0]
    print(f"📥 Chargement : {csv_file.name}")
    print(f"   Taille : {csv_file.stat().st_size / 1e6:.2f} MB")
    
    # Charger échantillon pour identifier structure
    df_sample = pd.read_csv(csv_file, nrows=1000, sep=';', low_memory=False)
    print(f"\n📊 Colonnes disponibles :")
    print(list(df_sample.columns))
    print(f"\nÉchantillon :")
    print(df_sample.head())

📥 Chargement : DS_ETAT_CIVIL_NAIS_COMMUNES_data.csv
   Taille : 25.61 MB

📊 Colonnes disponibles :
['EC_MEASURE', 'FREQ', 'GEO', 'GEO_OBJECT', 'OBS_STATUS', 'TIME_PERIOD', 'OBS_VALUE']

Échantillon :
  EC_MEASURE FREQ    GEO GEO_OBJECT OBS_STATUS  TIME_PERIOD  OBS_VALUE
0        LVB    A  68170        COM          A         2018          5
1        LVB    A  68185        COM          A         2015         14
2        LVB    A  68181        COM          A         2018          1
3        LVB    A  68177        COM          A         2021          9
4        LVB    A  68191        COM          A         2011          4


In [8]:
# Identifier colonnes clés
# Format attendu : code commune + année + compteurs naissances

if csv_files:
    print("🔍 Identification colonnes clés...")
    
    # Chercher colonne code géographique
    geo_cols = [c for c in df_sample.columns if any(k in c.lower() for k in ['code', 'geo', 'commune', 'com'])]
    print(f"\nColonnes géographiques potentielles : {geo_cols}")
    
    # Chercher colonne année
    year_cols = [c for c in df_sample.columns if any(k in c.lower() for k in ['annee', 'an', 'year'])]
    print(f"Colonnes temporelles potentielles : {year_cols}")
    
    # Afficher types
    print(f"\nTypes de données :")
    print(df_sample.dtypes)

🔍 Identification colonnes clés...

Colonnes géographiques potentielles : ['GEO', 'GEO_OBJECT']
Colonnes temporelles potentielles : []

Types de données :
EC_MEASURE     object
FREQ           object
GEO             int64
GEO_OBJECT     object
OBS_STATUS     object
TIME_PERIOD     int64
OBS_VALUE       int64
dtype: object


In [9]:
# Charger données complètes et filtrer PC
# ATTENTION : Adapter colonnes selon structure réelle

if csv_files:
    print("📥 Chargement complet naissances...")
    
    # TODO: Adapter selon colonnes réelles identifiées ci-dessus
    # Exemple supposé : colonnes 'CODGEO', 'ANAIS', 'NBNAISS'
    
    # Pour l'instant, charger avec dtype object pour inspecter
    df_naissances_raw = pd.read_csv(csv_file, sep=';', low_memory=False, dtype=str)
    
    print(f"✅ {len(df_naissances_raw):,} lignes chargées")
    
    # Identifier colonne code commune (adapter selon résultats ci-dessus)
    # Ici on suppose 'CODGEO' ou similaire
    if 'CODGEO' in df_naissances_raw.columns:
        df_naissances_raw['code_insee'] = df_naissances_raw['CODGEO'].str.zfill(5)
    elif 'COM' in df_naissances_raw.columns:
        df_naissances_raw['code_insee'] = df_naissances_raw['COM'].str.zfill(5)
    else:
        # Chercher automatiquement première colonne ressemblant à un code
        for col in df_naissances_raw.columns:
            if df_naissances_raw[col].str.match(r'^\d{2,5}$').sum() > len(df_naissances_raw) * 0.5:
                df_naissances_raw['code_insee'] = df_naissances_raw[col].str.zfill(5)
                print(f"⚠️  Colonne code INSEE détectée automatiquement : {col}")
                break
    
    # Filtrer Petite Couronne
    if 'code_insee' in df_naissances_raw.columns:
        df_naissances_pc = df_naissances_raw[df_naissances_raw['code_insee'].isin(codes_insee_pc)].copy()
        print(f"🎯 Après filtrage PC : {len(df_naissances_pc):,} lignes")
    else:
        print("❌ Impossible de standardiser code INSEE")
        df_naissances_pc = None

📥 Chargement complet naissances...
✅ 710,821 lignes chargées
⚠️  Colonne code INSEE détectée automatiquement : GEO
🎯 Après filtrage PC : 2,448 lignes


In [10]:
# Agréger par commune-année
# ATTENTION : Adapter selon structure réelle

if csv_files and df_naissances_pc is not None:
    print("🔄 Agrégation par commune-année...")
    
    # Identifier colonne année
    year_col = None
    for col in ['ANAIS', 'ANNEE', 'AN']:
        if col in df_naissances_pc.columns:
            year_col = col
            break
    
    if year_col:
        df_naissances_pc['annee'] = pd.to_numeric(df_naissances_pc[year_col], errors='coerce')
        
        # Compter naissances par commune-année
        df_naissances_agg = df_naissances_pc.groupby(['code_insee', 'annee']).size().reset_index(name='nb_naissances')
        
        print(f"✅ {len(df_naissances_agg):,} lignes agrégées")
        print(f"\nPériode : {df_naissances_agg['annee'].min():.0f} - {df_naissances_agg['annee'].max():.0f}")
        print(f"\nÉchantillon :")
        print(df_naissances_agg.head(10))
        
        # Sauvegarder en Silver
        output_naissances = DATA_SILVER / "naissances_pc_2008_2024.parquet"
        df_naissances_agg.to_parquet(output_naissances, index=False)
        print(f"\n✅ Sauvegardé : {output_naissances.name}")
        print(f"   Taille : {output_naissances.stat().st_size / 1024:.2f} KB")
    else:
        print("❌ Colonne année non trouvée")
        print(f"Colonnes disponibles : {list(df_naissances_pc.columns)}")

🔄 Agrégation par commune-année...
❌ Colonne année non trouvée
Colonnes disponibles : ['EC_MEASURE', 'FREQ', 'GEO', 'GEO_OBJECT', 'OBS_STATUS', 'TIME_PERIOD', 'OBS_VALUE', 'code_insee']


## 4️⃣ Décès 2008-2024

In [11]:
# Même logique que naissances
deces_dir = DATA_BRONZE / "deces_2008_2024"
csv_files_deces = list(deces_dir.glob("*.csv"))

if not csv_files_deces:
    print("❌ Aucun fichier CSV décès trouvé")
else:
    csv_file_deces = csv_files_deces[0]
    print(f"📥 Chargement : {csv_file_deces.name}")
    print(f"   Taille : {csv_file_deces.stat().st_size / 1e6:.2f} MB")
    
    # Charger avec dtype object
    df_deces_raw = pd.read_csv(csv_file_deces, sep=';', low_memory=False, dtype=str)
    
    print(f"✅ {len(df_deces_raw):,} lignes chargées")
    print(f"\nColonnes : {list(df_deces_raw.columns)}")
    print(f"\nÉchantillon :")
    print(df_deces_raw.head())

📥 Chargement : DS_ETAT_CIVIL_DECES_COMMUNES_data.csv
   Taille : 25.59 MB
✅ 710,821 lignes chargées

Colonnes : ['EC_MEASURE', 'FREQ', 'GEO', 'GEO_OBJECT', 'OBS_STATUS', 'TIME_PERIOD', 'OBS_VALUE']

Échantillon :
  EC_MEASURE FREQ    GEO GEO_OBJECT OBS_STATUS TIME_PERIOD OBS_VALUE
0        DTH    A  97602        COM          M        2010       NaN
1        DTH    A  97602        COM          M        2011       NaN
2        DTH    A  97602        COM          M        2012       NaN
3        DTH    A  97602        COM          M        2013       NaN
4        DTH    A  97603        COM          M        2009       NaN


In [12]:
# Filtrer et agréger décès
if csv_files_deces:
    print("🔄 Traitement décès...")
    
    # Standardiser code INSEE
    if 'CODGEO' in df_deces_raw.columns:
        df_deces_raw['code_insee'] = df_deces_raw['CODGEO'].str.zfill(5)
    elif 'COM' in df_deces_raw.columns:
        df_deces_raw['code_insee'] = df_deces_raw['COM'].str.zfill(5)
    else:
        # Auto-détection
        for col in df_deces_raw.columns:
            if df_deces_raw[col].str.match(r'^\d{2,5}$').sum() > len(df_deces_raw) * 0.5:
                df_deces_raw['code_insee'] = df_deces_raw[col].str.zfill(5)
                print(f"⚠️  Colonne code INSEE détectée : {col}")
                break
    
    # Filtrer PC
    if 'code_insee' in df_deces_raw.columns:
        df_deces_pc = df_deces_raw[df_deces_raw['code_insee'].isin(codes_insee_pc)].copy()
        print(f"🎯 Après filtrage PC : {len(df_deces_pc):,} lignes")
        
        # Identifier colonne année
        year_col = None
        for col in ['ANDEC', 'ANNEE', 'AN']:
            if col in df_deces_pc.columns:
                year_col = col
                break
        
        if year_col:
            df_deces_pc['annee'] = pd.to_numeric(df_deces_pc[year_col], errors='coerce')
            
            # Agréger
            df_deces_agg = df_deces_pc.groupby(['code_insee', 'annee']).size().reset_index(name='nb_deces')
            
            print(f"✅ {len(df_deces_agg):,} lignes agrégées")
            print(f"\nPériode : {df_deces_agg['annee'].min():.0f} - {df_deces_agg['annee'].max():.0f}")
            print(f"\nÉchantillon :")
            print(df_deces_agg.head(10))
            
            # Sauvegarder
            output_deces = DATA_SILVER / "deces_pc_2008_2024.parquet"
            df_deces_agg.to_parquet(output_deces, index=False)
            print(f"\n✅ Sauvegardé : {output_deces.name}")
            print(f"   Taille : {output_deces.stat().st_size / 1024:.2f} KB")
        else:
            print("❌ Colonne année non trouvée")
    else:
        print("❌ Impossible de standardiser code INSEE")

🔄 Traitement décès...
⚠️  Colonne code INSEE détectée : GEO
🎯 Après filtrage PC : 2,448 lignes
❌ Colonne année non trouvée


## 5️⃣ Élections Agrégées 1999-2024

In [13]:
# Fichier volumineux : charger par chunks
elections_path = DATA_BRONZE / "elections_agregees_1999_2024.csv"

if not elections_path.exists():
    print("❌ Fichier élections non trouvé")
else:
    print(f"📥 Chargement élections (2.35 GB)...")
    print("   (en chunks pour optimiser mémoire)")
    
    # Charger échantillon pour structure
    df_elec_sample = pd.read_csv(elections_path, nrows=1000, sep=';', low_memory=False)
    print(f"\n📊 Colonnes :")
    print(list(df_elec_sample.columns))
    print(f"\nÉchantillon :")
    print(df_elec_sample.head())

📥 Chargement élections (2.35 GB)...
   (en chunks pour optimiser mémoire)

📊 Colonnes :
['id_election', 'id_brut_miom', 'code_departement', 'code_commune', 'code_bv', 'no_panneau', 'voix', 'ratio_voix_inscrits', 'ratio_voix_exprimes', 'nuance', 'sexe', 'nom', 'prenom', 'liste', 'libelle_abrege_liste', 'libelle_etendu_liste', 'nom_tete_liste', 'binome']

Échantillon :
    id_election id_brut_miom  code_departement  code_commune  code_bv  \
0  2022_pres_t1   01001_0001                 1          1001        1   
1  2022_pres_t1   01002_0001                 1          1002        1   
2  2022_pres_t1   01004_0001                 1          1004        1   
3  2022_pres_t1   01004_0002                 1          1004        2   
4  2022_pres_t1   01004_0003                 1          1004        3   

   no_panneau  voix  ratio_voix_inscrits  ratio_voix_exprimes  nuance sexe  \
0           1     3                 0.47                 0.58     NaN    F   
1           1     2                

In [14]:
# Filtrer PC par chunks
if elections_path.exists():
    print("🔄 Filtrage Petite Couronne par chunks...")
    
    # Identifier colonne code commune
    code_col_elec = None
    for col in ['code_commune', 'code_insee', 'COM', 'CODGEO']:
        if col in df_elec_sample.columns:
            code_col_elec = col
            break
    
    if code_col_elec is None:
        print("❌ Colonne code commune non trouvée")
        print(f"Colonnes disponibles : {list(df_elec_sample.columns)}")
    else:
        print(f"✅ Colonne code commune : {code_col_elec}")
        
        # Charger par chunks et filtrer
        chunks_pc = []
        chunk_size = 100000
        
        for i, chunk in enumerate(pd.read_csv(elections_path, sep=';', chunksize=chunk_size, low_memory=False)):
            # Standardiser code INSEE
            chunk['code_insee'] = chunk[code_col_elec].astype(str).str.zfill(5)
            
            # Filtrer PC
            chunk_pc = chunk[chunk['code_insee'].isin(codes_insee_pc)]
            
            if len(chunk_pc) > 0:
                chunks_pc.append(chunk_pc)
            
            if (i + 1) % 10 == 0:
                print(f"   Traité {(i+1) * chunk_size:,} lignes...")
        
        # Concaténer chunks
        if chunks_pc:
            df_elec_pc = pd.concat(chunks_pc, ignore_index=True)
            print(f"\n✅ {len(df_elec_pc):,} lignes PC retenues")
            
            # Forcer types string pour colonnes codes (éviter mixage int/string)
            print("\n🔧 Nettoyage types pour Parquet...")
            for col in df_elec_pc.columns:
                if 'code' in col.lower() or 'dep' in col.lower():
                    df_elec_pc[col] = df_elec_pc[col].astype(str)
            
            # Sauvegarder
            output_elec = DATA_SILVER / "elections_pc_1999_2024.parquet"
            df_elec_pc.to_parquet(output_elec, index=False)
            print(f"✅ Sauvegardé : {output_elec.name}")
            print(f"   Taille : {output_elec.stat().st_size / 1e6:.2f} MB")
        else:
            print("❌ Aucune donnée PC trouvée")

🔄 Filtrage Petite Couronne par chunks...
✅ Colonne code commune : code_commune
   Traité 1,000,000 lignes...
   Traité 2,000,000 lignes...
   Traité 3,000,000 lignes...
   Traité 4,000,000 lignes...
   Traité 5,000,000 lignes...
   Traité 6,000,000 lignes...
   Traité 7,000,000 lignes...
   Traité 8,000,000 lignes...
   Traité 9,000,000 lignes...
   Traité 10,000,000 lignes...
   Traité 11,000,000 lignes...
   Traité 12,000,000 lignes...
   Traité 13,000,000 lignes...
   Traité 14,000,000 lignes...
   Traité 15,000,000 lignes...
   Traité 16,000,000 lignes...
   Traité 17,000,000 lignes...
   Traité 18,000,000 lignes...
   Traité 19,000,000 lignes...
   Traité 20,000,000 lignes...
   Traité 21,000,000 lignes...
   Traité 22,000,000 lignes...
   Traité 23,000,000 lignes...
   Traité 24,000,000 lignes...
   Traité 25,000,000 lignes...
   Traité 26,000,000 lignes...
   Traité 27,000,000 lignes...

✅ 1,517,115 lignes PC retenues

🔧 Nettoyage types pour Parquet...
✅ Sauvegardé : elections_p

## 6️⃣ Revenus Communes

In [15]:
# Revenus par commune
revenus_path = DATA_BRONZE / "revenus_commune.csv"

if not revenus_path.exists():
    print("❌ Fichier revenus non trouvé")
else:
    print("📥 Chargement revenus...")
    df_revenus = pd.read_csv(revenus_path, sep=';', low_memory=False)
    
    print(f"✅ {len(df_revenus):,} lignes chargées")
    print(f"\nColonnes : {list(df_revenus.columns)}")
    print(f"\nÉchantillon :")
    print(df_revenus.head())

📥 Chargement revenus...
✅ 34,926 lignes chargées

Colonnes : ['Nom géographique GMS', 'Code géographique', 'Libellé géographique', '[DISP] Nbre de ménages fiscaux', '[DISP] Nbre de personnes dans les ménages fiscaux', "[DISP] Nbre d'unités de consommation dans les ménages fiscaux", '[DISP] 1ᵉʳ quartile (€)', '[DISP] Médiane (€)', '[DISP] 3ᵉ quartile (€)', '[DISP] Écart interquartile (€)', '[DISP] 1ᵉʳ décile (€)', '[DISP] 2ᵉ décile (€)', '[DISP]3ᵉ décile (€)', '[DISP] 4ᵉ décile (€)', '[DISP] 6ᵉ décile (€)', '[DISP] 7ᵉ décile (€)', '[DISP] 8ᵉ décile (€)', '[DISP] 9ᵉ décile (€)', '[DISP] Rapport interdécile 9ᵉ décile/1ᵉʳ decile', '[DISP] S80/20', '[DISP] Iice de Gini', '[DISP] Part des revenus d’activité (%)', '[DISP] dont part des salaires et traitements(%)', '[DISP] dont part des iemnités de chômage (%)', '[DISP] dont part des revenus des activités non salariées (%)', '[DISP] Part des pensions, retraites et rentes (%)', '[DISP] Part des revenus du patrimoine et autres revenus (%)', "[DI

In [16]:
# Filtrer PC
if revenus_path.exists():
    print("🔄 Filtrage revenus PC...")
    
    # Identifier colonne code
    code_col_rev = None
    for col in ['CODGEO', 'code_commune', 'code_insee', 'COM']:
        if col in df_revenus.columns:
            code_col_rev = col
            break
    
    if code_col_rev:
        df_revenus['code_insee'] = df_revenus[code_col_rev].astype(str).str.zfill(5)
        df_revenus_pc = df_revenus[df_revenus['code_insee'].isin(codes_insee_pc)].copy()
        
        print(f"✅ {len(df_revenus_pc):,} lignes PC retenues")
        
        # Sauvegarder
        output_rev = DATA_SILVER / "revenus_pc.parquet"
        df_revenus_pc.to_parquet(output_rev, index=False)
        print(f"✅ Sauvegardé : {output_rev.name}")
        print(f"   Taille : {output_rev.stat().st_size / 1024:.2f} KB")
    else:
        print("❌ Colonne code non trouvée")

🔄 Filtrage revenus PC...
❌ Colonne code non trouvée


## 7️⃣ Validation Qualité Silver

In [17]:
# Lister fichiers Silver créés
print("📊 INVENTAIRE DONNÉES SILVER")
print("=" * 60)

silver_files = sorted(DATA_SILVER.glob("*.parquet"))
total_size = 0

for f in silver_files:
    size_mb = f.stat().st_size / 1e6
    total_size += size_mb
    print(f"✅ {f.name:50s} {size_mb:8.2f} MB")

print("=" * 60)
print(f"TOTAL : {total_size:.2f} MB")

📊 INVENTAIRE DONNÉES SILVER
✅ elections_pc_1999_2024.parquet                         8.76 MB
✅ referentiel_communes_pc_2024.parquet                   0.00 MB
✅ referentiel_petite_couronne.parquet                    0.00 MB
TOTAL : 8.77 MB


In [18]:
# Vérifier cohérence géographique
print("\n🔍 VALIDATION GÉOGRAPHIQUE")
print("=" * 60)

expected_communes = len(codes_insee_pc)
print(f"Communes attendues (COG 2024) : {expected_communes}")

# Vérifier chaque dataset
for f in silver_files:
    if 'referentiel' not in f.name:
        df_check = pd.read_parquet(f)
        if 'code_insee' in df_check.columns:
            n_communes = df_check['code_insee'].nunique()
            coverage = (n_communes / expected_communes) * 100
            status = "✅" if coverage > 90 else "⚠️"
            print(f"{status} {f.name:40s} : {n_communes:3d} communes ({coverage:5.1f}%)")


🔍 VALIDATION GÉOGRAPHIQUE
Communes attendues (COG 2024) : 144
⚠️ elections_pc_1999_2024.parquet           : 124 communes ( 86.1%)


In [19]:
# Vérifier couverture temporelle
print("\n📅 VALIDATION TEMPORELLE")
print("=" * 60)

for f in silver_files:
    if 'referentiel' not in f.name:
        df_check = pd.read_parquet(f)
        if 'annee' in df_check.columns:
            min_year = df_check['annee'].min()
            max_year = df_check['annee'].max()
            n_years = df_check['annee'].nunique()
            print(f"📊 {f.name:40s} : {min_year:.0f}-{max_year:.0f} ({n_years} années)")


📅 VALIDATION TEMPORELLE


## 8️⃣ Synthèse ETL

In [20]:
print("\n" + "=" * 60)
print("📋 SYNTHÈSE ETL BRONZE → SILVER")
print("=" * 60)

print("\n✅ TRANSFORMATIONS APPLIQUÉES :")
print("  1. Filtrage géographique Petite Couronne (75, 92, 93, 94)")
print("  2. Standardisation codes INSEE (5 caractères)")
print("  3. Création clé jointure 'code_insee'")
print("  4. Agrégation naissances/décès par commune-année")
print("  5. Format parquet optimisé")

print("\n🎯 DATASETS SILVER DISPONIBLES :")
for f in silver_files:
    print(f"  - {f.name}")

print("\n📊 MÉTRIQUES QUALITÉ :")
print(f"  - Communes Petite Couronne : {expected_communes}")
print(f"  - Départements : {', '.join(DEPARTEMENTS_PC)}")
print(f"  - Taille totale Silver : {total_size:.2f} MB")

print("\n🚀 PROCHAINE ÉTAPE :")
print("  Notebook 05_feature_engineering.ipynb")
print("  - Jointure tous datasets sur code_insee + annee")
print("  - Création features ML (ratios, tendances, etc.)")
print("  - Préparation variable cible (Gauche/Centre/Droite)")
print("  - Export vers couche Gold")


📋 SYNTHÈSE ETL BRONZE → SILVER

✅ TRANSFORMATIONS APPLIQUÉES :
  1. Filtrage géographique Petite Couronne (75, 92, 93, 94)
  2. Standardisation codes INSEE (5 caractères)
  3. Création clé jointure 'code_insee'
  4. Agrégation naissances/décès par commune-année
  5. Format parquet optimisé

🎯 DATASETS SILVER DISPONIBLES :
  - elections_pc_1999_2024.parquet
  - referentiel_communes_pc_2024.parquet
  - referentiel_petite_couronne.parquet

📊 MÉTRIQUES QUALITÉ :
  - Communes Petite Couronne : 144
  - Départements : 75, 92, 93, 94
  - Taille totale Silver : 8.77 MB

🚀 PROCHAINE ÉTAPE :
  Notebook 05_feature_engineering.ipynb
  - Jointure tous datasets sur code_insee + annee
  - Création features ML (ratios, tendances, etc.)
  - Préparation variable cible (Gauche/Centre/Droite)
  - Export vers couche Gold
